# RAG-03 · Measured Baseline RAG Retrieval
### Section 06 — Establish evidence before infrastructure

**Prerequisites:** NLP-01, NLP-02, RAG-01, and RAG-02.
**Estimated time:** 6–8 hours.
This notebook creates the measurement baseline that vector databases, hybrid
search, reranking, and advanced RAG must beat. It evaluates retrieval only;
generation quality is a separate downstream problem.


> **Canonical learner route · module RAG-03 of 65**
>
> Required prerequisites: **NLP-01, NLP-02, RAG-01, RAG-02** · Previous: **RAG-02** ·
> Next after mastery: **RAG-04** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

Version corpus and labels, preserve evidence provenance, compare chunking,
implement lexical and dense statistical retrieval, fuse rankings, calculate
recall@k/MRR/nDCG, measure abstention and latency, and diagnose failure slices.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · RAG-03 · Measured Baseline RAG Retrieval

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


## 2 · Historical Motivation

RAG systems often added embeddings, vector stores, rerankers, and agents before
anyone established whether retrieval improved. Classical information retrieval
starts with a test collection: documents, queries, relevance judgements, metrics,
and a baseline. RAG needs the same discipline.


## 3 · Intuition and Visual Understanding

```text
versioned documents → chunks with evidence IDs → retriever → ranked chunks
         ↑                                              ↓
   labelled queries ← failure review ← metrics and slices
```

If the needed evidence is absent, generation cannot be grounded. If evidence is
present and the answer is wrong, the generator failed. Keep these diagnoses apart.


## 4 · Mathematical Foundations

Recall@k is the fraction of relevant evidence retrieved in the first k results.
Reciprocal rank is $1/r$ where $r$ is the first relevant rank. DCG discounts a
relevant result at rank $i$ by $1/\log_2(i+1)$; nDCG divides by the ideal DCG.

**Small example:** if the only relevant section is ranked third, recall@3 is 1,
reciprocal rank is 1/3, and nDCG@3 is $1/\log_2(4)=0.5$. Recall says it was found;
rank-sensitive metrics say the learner had to look past two distractions.


## 5 · Manual Implementation from Scratch

Run the project evaluation. The dense baseline is LSA—TF-IDF projected with
truncated SVD—not a neural sentence embedding. That label matters.


In [ ]:
import sys
from pathlib import Path
ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "projects/rag_foundations/src").exists()
)
sys.path.insert(0, str(ROOT / "projects/rag_foundations/src"))
from rag_foundations.evaluation import evaluate

report = evaluate(ROOT / "projects/rag_foundations/data")
for name, result in report["experiments"].items():
    print(name, "recall", round(result["recall_at_k"], 3),
          "MRR", round(result["mrr"], 3),
          "nDCG", round(result["ndcg_at_k"], 3))


## 6 · Visualization

Compare systems on more than one axis. A small ranking improvement may not justify
worse abstention, latency, operational complexity, or failure on paraphrases.


In [ ]:
import matplotlib.pyplot as plt
names = list(report["experiments"])
recalls = [report["experiments"][name]["recall_at_k"] for name in names]
plt.figure(figsize=(10, 4)); plt.bar(range(len(names)), recalls)
plt.xticks(range(len(names)), names, rotation=60, ha="right")
plt.ylabel("Recall@5"); plt.ylim(0, 1.05); plt.tight_layout(); plt.show()


## 7 · Failure Modes and Common Mistakes

- Labelling queries after seeing one system's results.
- Losing source/section IDs during chunking.
- Calling LSA a neural semantic encoder.
- Reporting only an average and hiding paraphrase failures.
- Treating no lexical overlap as proof a question is unanswerable.
- Adding raw scores from different retrievers rather than fusing ranks.
- Evaluating generated answers when retrieval evidence was never measured.


## 8 · Library Implementation

The project uses sklearn TF-IDF and TruncatedSVD, then deterministic ranking and
metric functions. Inspect `projects/rag_foundations/src/rag_foundations/evaluation.py`.
Every report includes corpus/query hashes and per-query component rows.


## 9 · Realistic Case Study

The corpus contains curriculum knowledge with direct, paraphrased, multi-concept,
and unanswerable questions. The best aggregate system is not automatically the
deployment choice; inspect which student questions it misses and why.


## 10 · Production and Learning Considerations

Version documents, parsers, chunks, query labels, representation, and thresholds.
Log evidence IDs, scores, latency, and abstention for every query. Re-evaluate
whenever corpus, chunker, or embedding model changes.


## 11 · Tradeoff Analysis

Lexical retrieval is transparent and excellent for exact terms. LSA captures
latent co-occurrence but is corpus-dependent. Rank fusion is robust to score scale
but adds another choice. Structure chunks preserve meaning; smaller chunks may rank
precisely while losing surrounding context.


## 12 · Readiness and Interview Preparation

Given one failed query, decide whether the label, chunking, representation,
ranking, threshold, or corpus is responsible. Propose one change and one metric
that would confirm or falsify the diagnosis.


## 13 · Teach-Back

Explain why retrieval evaluation precedes generation. Contrast recall@k, MRR,
and nDCG. Explain why an unanswerable set and evidence provenance are safety
requirements, not optional metrics.


## 14 · Exercises, Self-Check, and Solutions

**Worked:** manually rank three sections for one query and calculate all metrics.
**Guided:** inspect three component failures and assign a cause. **Independent:**
add two queries without changing corpus text, predict which retriever wins, then
evaluate. **Challenge:** add a pinned neural embedding model as a new experiment
without removing lexical/LSA baselines or changing gold labels.

<details><summary><strong>Solution and scoring rubric</strong></summary>
Full credit preserves IDs and hashes, separates retrieval from answer claims,
reports slices, and makes one-variable ablations. Award 3 points for metric math,
3 for diagnosis, 2 for reproducibility, and 2 for limitations. Common mistakes:
changing labels to improve scores and claiming the most complex system must win.
**Readiness threshold: 8/10.**
</details>


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **RAG-03 · Measured Baseline RAG Retrieval** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember RAG-03 · Measured Baseline RAG Retrieval: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · RAG-03

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
